In [5]:
from pathlib import Path
import time
import pandas as pd
from Bio import SeqIO

FASTA = Path("../data/PanTEon_Database_v1.6.1.fasta")

filas = []
mal_formados = []
t0 = time.time()

for rec in SeqIO.parse(FASTA, "fasta"):
    header = rec.description
    raw = str(rec.seq)
    seq = raw.upper()
    L = len(seq)
    n_N = seq.count("N")
    otros = L - sum(seq.count(x) for x in "ACGTN")
    raros = "".join(sorted(set(seq) - set("ACGTN"))) if otros else ""

    if "#" not in header or "@" not in header:
        mal_formados.append(header)
        continue

    id_, resto = header.split("#", 1)
    clasif, especie = resto.split("@", 1)

    filas.append({
        "id": id_.strip(),
        "clasificacion": clasif.strip(),
        "niveles": clasif.count("/") + 1,
        "especie": especie.strip(),
        "longitud": L,
        "n_N": n_N,
        "otros": otros,
        "raros": raros,
        "tiene_mayus": raw != raw.lower(),
        "tiene_minus": raw != raw.upper(),
    })

print(f"Listo en {time.time() - t0:.0f} s")

Listo en 30 s


In [7]:
fa = pd.DataFrame(filas)
print("Registros leídos:", len(fa) + len(mal_formados))
print("Encabezados mal formados:", len(mal_formados))
print(fa.shape)
fa.head()

Registros leídos: 239946
Encabezados mal formados: 0
(239946, 10)


,id,clasificacion,niveles,especie,longitud,n_N,otros,raros,tiene_mayus,tiene_minus
0,PDB00000001,CLASSII/HELITRON/HELITRON,3,Halicephalobus mephisto,1991,0,0,,False,True
1,PDB00000002,CLASSII/HELITRON/HELITRON,3,Halicephalobus mephisto,1689,0,0,,False,True
2,PumCon-1.11,CLASSI/SINE/TRNA,3,Puma concolor,235,0,0,,True,False
3,PumCon-1.141,CLASSII/TIR/TC1MARINER,3,Puma concolor,340,0,0,,True,False
4,PumCon-1.14,CLASSI/SINE/TRNA,3,Puma concolor,233,0,0,,True,False


In [8]:
fa.to_csv("../data/fasta_resumen.csv", index=False)
fa["longitud"].describe()

count    239946.000000
mean       4017.159757
std        2751.326679
min          81.000000
25%        1463.000000
50%        4259.000000
75%        5656.000000
max       19926.000000
Name: longitud, dtype: float64

In [9]:
print("Especies distintas:", fa["especie"].nunique())
print("IDs distintos:", fa["id"].nunique(), "de", len(fa))
print("IDs duplicados:", fa["id"].duplicated().sum())
print("Con prefijo PDB:", fa["id"].str.startswith("PDB").sum())
print("Niveles de clasificación:\n", fa["niveles"].value_counts())
print("Solo minúsculas:", (fa["tiene_minus"] & ~fa["tiene_mayus"]).sum())
print("Solo mayúsculas:", (fa["tiene_mayus"] & ~fa["tiene_minus"]).sum())
print("Mezcladas:", (fa["tiene_mayus"] & fa["tiene_minus"]).sum())
print("Con alguna N:", (fa["n_N"] > 0).sum())
print("Con otros caracteres:", (fa["otros"] > 0).sum())

Especies distintas: 2790
IDs distintos: 239946 de 239946
IDs duplicados: 0
Con prefijo PDB: 208346
Niveles de clasificación:
 niveles
3    239946
Name: count, dtype: int64
Solo minúsculas: 208354
Solo mayúsculas: 31592
Mezcladas: 0
Con alguna N: 54837
Con otros caracteres: 0


In [10]:
fa["pct_N"] = fa["n_N"] / fa["longitud"] * 100
con_N = fa[fa["n_N"] > 0]
print(con_N["pct_N"].describe())
print("Más de 5% de N:", (fa["pct_N"] > 5).sum())
print("Más de 50% de N:", (fa["pct_N"] > 50).sum())
print(pd.crosstab(fa["id"].str.startswith("PDB"), fa["tiene_minus"]))

count    54837.000000
mean         1.658556
std          2.008501
min          0.006398
25%          0.334200
50%          0.913242
75%          2.173913
max         50.062735
Name: pct_N, dtype: float64
Más de 5% de N: 4374
Más de 50% de N: 1
tiene_minus  False   True 
id                        
False        31592       8
True             0  208346


In [11]:
print(fa.loc[fa["pct_N"] > 50, ["id","especie","clasificacion","longitud","n_N","pct_N"]])
raras = fa[~fa["id"].str.startswith("PDB") & fa["tiene_minus"]]
print(raras[["id","especie","clasificacion","longitud"]])

                                  id            especie  \
5055  Halyomorpha_halys-5_family-957  Halyomorpha halys   

               clasificacion  longitud  n_N      pct_N  
5055  CLASSII/TIR/TC1MARINER       797  399  50.062735  
                      id                    especie     clasificacion  \
69805     Chlamys-2_cRei  Chlamydomonas reinhardtii    CLASSI/PLE/PLE   
69806    Chlamys-5b_cRei  Chlamydomonas reinhardtii    CLASSI/PLE/PLE   
69807     Chlamys-7_cRei  Chlamydomonas reinhardtii    CLASSI/PLE/PLE   
69808    Chlamys-N4_cRei  Chlamydomonas reinhardtii    CLASSI/PLE/PLE   
69809    Chlamys-N6_cRei  Chlamydomonas reinhardtii    CLASSI/PLE/PLE   
101711  Chlamys-10b_cRei  Chlamydomonas reinhardtii  CLASSI/LINE/LINE   
101966  Chlamys-10c_cRei  Chlamydomonas reinhardtii  CLASSI/LINE/LINE   
102221  Chlamys-N2a_cRei  Chlamydomonas reinhardtii  CLASSI/LINE/LINE   

        longitud  
69805       3081  
69806        887  
69807       2079  
69808       1367  
69809        1

In [12]:
csv = pd.read_csv("../data/PanTEon_Database_metadata_v1.6.1.csv", dtype=str, keep_default_na=False)
sp_fa, sp_csv = set(fa["especie"]), set(csv["Species"])

print("En FASTA y no en CSV:", len(sp_fa - sp_csv))
print("En CSV y no en FASTA:", len(sp_csv - sp_fa))
print(sorted(sp_fa - sp_csv)[:10])
print(sorted(sp_csv - sp_fa)[:10])

csv[csv["Species"] == "Chlamydomonas reinhardtii"][["Species", "Source", "Source 2", "Notes"]]

En FASTA y no en CSV: 4
En CSV y no en FASTA: 4
['Abrus pulchellus subsp. cantoniensis', 'Beta vulgaris subsp. vulgaris', 'Coccomyxa sp. obi', 'Marchantia polymorpha subsp. montivagans']
['Abrus pulchellus subsp cantoniensis', 'Beta vulgaris subsp vulgaris', 'Coccomyxa sp obi', 'Marchantia polymorpha subsp montivagans']


,Species,Source,Source 2,Notes
708,Chlamydomonas reinhardtii,RM2+MCH,Repbase,Uncurated Dfam


In [13]:
import re

def clave(s):
    s = s.strip().lower().replace(".", "")
    return re.sub(r"\s+", " ", s)

fa["clave"] = fa["especie"].map(clave)
csv["clave"] = csv["Species"].map(clave)

print("FASTA: claves", fa["clave"].nunique(), "| nombres", fa["especie"].nunique())
print("CSV:   claves", csv["clave"].nunique(), "| nombres", csv["Species"].nunique())
print("Solo en FASTA:", set(fa["clave"]) - set(csv["clave"]))
print("Solo en CSV:", set(csv["clave"]) - set(fa["clave"]))

FASTA: claves 2790 | nombres 2790
CSV:   claves 2790 | nombres 2790
Solo en FASTA: set()
Solo en CSV: set()


In [14]:
import sys
from pathlib import Path

# Para que Python encuentre la carpeta pipeline desde notebooks/
sys.path.insert(0, str(Path("..").resolve()))

from pipeline.limpiar import clave, limpiar_csv

# Clave de cruce del lado FASTA (misma función y mismo nombre de columna que el CSV)
fa["species_key"] = fa["especie"].map(clave)

# CSV limpio, con su species_key
limpio = limpiar_csv("../data/PanTEon_Database_metadata_v1.6.1.csv")

# Cruce: especies sin pareja en cada lado
solo_fa = set(fa["species_key"]) - set(limpio["species_key"])
solo_csv = set(limpio["species_key"]) - set(fa["species_key"])

print("Claves FASTA:", fa["species_key"].nunique(), "| nombres:", fa["especie"].nunique())
print("Claves CSV:  ", limpio["species_key"].nunique(), "| nombres:", limpio["Species"].nunique())
print("Solo en FASTA:", solo_fa)
print("Solo en CSV:", solo_csv)

# Prueba de que el cruce asigna taxonomía a todas las secuencias
cruce = fa.merge(limpio, on="species_key", how="left", indicator=True)
print("Secuencias sin taxonomía:", (cruce["_merge"] != "both").sum())
print("Filas tras el cruce:", len(cruce), "| filas en FASTA:", len(fa))

Claves FASTA: 2790 | nombres: 2790
Claves CSV:   2790 | nombres: 2790
Solo en FASTA: set()
Solo en CSV: set()
Secuencias sin taxonomía: 0
Filas tras el cruce: 239946 | filas en FASTA: 239946


In [15]:
from pathlib import Path
import pandas as pd

Path("../reports").mkdir(exist_ok=True)


nom_fa = set(fa["especie"])
nom_csv = set(limpio["Species"])

solo_fa_antes = sorted(nom_fa - nom_csv)
solo_csv_antes = sorted(nom_csv - nom_fa)

print("ANTES de normalizar")
print("  Solo en FASTA:", len(solo_fa_antes))
print("  Solo en CSV:  ", len(solo_csv_antes))

# una fila por especie sin pareja
reporte = pd.DataFrame(
    [{"lado": "solo_fasta", "nombre_original": n, "species_key": clave(n)} for n in solo_fa_antes] +
    [{"lado": "solo_csv",   "nombre_original": n, "species_key": clave(n)} for n in solo_csv_antes]
)

claves_fa = set(fa["species_key"])
claves_csv = set(limpio["species_key"])
reporte["cruza_tras_normalizar"] = reporte["species_key"].isin(claves_fa & claves_csv)

reporte.to_csv("../reports/reporte_cruce.csv", index=False)
print(reporte.to_string())

# DESPUÉS de normalizar
print("\nDESPUÉS de normalizar")
print("  Solo en FASTA:", len(claves_fa - claves_csv))
print("  Solo en CSV:  ", len(claves_csv - claves_fa))

ANTES de normalizar
  Solo en FASTA: 4
  Solo en CSV:   4
         lado                           nombre_original                              species_key  cruza_tras_normalizar
0  solo_fasta      Abrus pulchellus subsp. cantoniensis      abrus pulchellus subsp cantoniensis                   True
1  solo_fasta             Beta vulgaris subsp. vulgaris             beta vulgaris subsp vulgaris                   True
2  solo_fasta                         Coccomyxa sp. obi                         coccomyxa sp obi                   True
3  solo_fasta  Marchantia polymorpha subsp. montivagans  marchantia polymorpha subsp montivagans                   True
4    solo_csv       Abrus pulchellus subsp cantoniensis      abrus pulchellus subsp cantoniensis                   True
5    solo_csv              Beta vulgaris subsp vulgaris             beta vulgaris subsp vulgaris                   True
6    solo_csv                          Coccomyxa sp obi                         coccomyxa sp obi      